Train a DCGAN to tackle the image dataset of your choice, and use it to
generate images. Add experience replay and see if this helps. Turn it
into a conditional GAN where you can control the generated class.

In [1]:
%pip install tensorflow_datasets

import tensorflow as tf
from tensorflow.keras import mixed_precision

print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# 1) Evitar que TF reserve toda la VRAM al inicio
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# 2) Mixed precision (muy recomendado en RTX modernas)
mixed_precision.set_global_policy("float32")

# 4) Estrategia: solo usar MirroredStrategy si hay >1 GPU
strategy = tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy()
print("Strategy:", type(strategy).__name__)
print("Policy:", mixed_precision.global_policy())


TF: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy: _DefaultDistributionStrategy
Policy: <DTypePolicy "float32">


In [12]:
import tensorflow_datasets as tdfs

train_dataset = tdfs.load(
    "celeb_a",
    with_info=False
)

batch_size = 128

def prep(ex):
    img = tf.cast(ex["image"], tf.float32) / 255.0
    return img   # <- input y target

train_ds = train_dataset["train"].map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

NonMatchingChecksumError: Artifact https://drive.google.com/uc?export=download&id=0B7EVK8r0v71pZjFTYXZWM3FlRnM, downloaded to /root/tensorflow_datasets/downloads/celeb_a/ucexport_download_id_0B7EVK8r0v71pZjFTYXZWM3FlDDaXUAQO8EGH_a7VqGNLRtW52mva1LzDrb-V723OQN8.tmp.a421deb88dd242fb99667a61c5dfacdd/download, has wrong checksum:
* Expected: UrlInfo(size=1.34 GiB, checksum='46fb89443c578308acf364d7d379fe1b9efb793042c0af734b6112e4fd3a8c74', filename='img_align_celeba.zip')
* Got: UrlInfo(size=1.96 KiB, checksum='4405eece6b7815d401419481f14ce84e338b7551d27625bebf6eea2d1982df4b', filename='download')
To debug, see: https://www.tensorflow.org/datasets/overview#fixing_nonmatchingchecksumerror

In [5]:
codings_size = 128

generator = tf.keras.Sequential([
    tf.keras.layers.Dense(55 * 45 * 3),
    tf.keras.layers.Reshape([55, 45, 3]),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2DTranspose(3, kernel_size=5, strides=2, padding="same", activation="tanh"),
    tf.keras.layers.Cropping2D(cropping=(1,1))
])

discriminator = tf.keras.Sequential([
        tf.keras.layers.Conv2D(64, kernel_size=5, strides=2, padding="same", activation="relu"),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Conv2D(128, kernel_size=5, strides=2, padding="same", activation=tf.keras.layers.LeakyReLU(0.2)),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1, activation="sigmoid")
])

gan = tf.keras.Sequential([generator, discriminator])

In [3]:
discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

In [4]:
def train_gan(gan, dataset, batch_size, codings_size, n_epochs):
    generator, discriminator = gan.layers
    for epoch in range(n_epochs):
        print(f"Epoch {epoch + 1}/{n_epochs}") 
        for batch in dataset:
            noise = tf.random.normal(shape=[batch_size, codings_size])
            generated_images = generator(noise)
            x_fake_and_real = tf.concat([generated_images, batch], axis=0)
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
            discriminator.train_on_batch(x_fake_and_real, y1)

            noise = tf.random.normal(shape=[batch_size, codings_size])
            y2 = tf.constant([[1.]] * batch_size)
            gan.train_on_batch(noise, y2)

train_gan(gan, train_ds, batch_size, codings_size, n_epochs=10)

NameError: name 'train_ds' is not defined

In [ ]:
codings = tf.random.normal(shape=[batch_size, codings_size])
generated_images = generator.predict(codings)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step


: 

: 

: 

: 

: 

: 

: 

In [ ]:
import numpy as np
import matplotlib as plt

# Pasar a numpy y reescalar si el generador usa tanh ([-1, 1])
images = generated_images[:16]
if hasattr(images, "numpy"):
    images = images.numpy()
images = np.clip(images, 0.0, 1.0)
figure, axes = plt.subplots(4, 4, figsize=(8, 8))
for index, axis in enumerate(axes.flat):
    if index < 16:
        axis.imshow(images[index])
    axis.axis("off")
plt.tight_layout()
plt.show()

: 

: 

: 

: 

: 

: 

: 